In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("Libraries imported successfully!")


Libraries imported successfully!


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

# Change this path to your actual Google Drive folder
DATA_DIR = "/content/drive/MyDrive/ML_project_dataset"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
submission = pd.read_csv(f"{DATA_DIR}/submission.csv")

print("Train:", train.shape)
print("Test:", test.shape)
print("Submission template:", submission.shape)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train: (18076, 12)
Test: (4520, 11)
Submission template: (4520, 2)


In [ ]:

# Recreate the engineered feature from the original notebook
bins = [0, 7, 14, 21, 30, 45, 100]
labels = ["1-7", "8-14", "15-21", "22-30", "31-45", "46+"]

for df in [train, test]:
    df["days_left_group"] = pd.cut(
        df["days_left"],
        bins=bins,
        labels=labels
    )

# Target
y = train["price"]

# Exclude target, identifier, and flight for now
excluded_columns = ["price", "id", "flight"]

X = train.drop(columns=excluded_columns)
X_test = test.drop(columns=["id", "flight"])

# Identify numeric and categorical features
numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

# Use exactly the same validation split as before
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("Training split:", X_train.shape)
print("Validation split:", X_val.shape)
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


Training split: (14460, 10)
Validation split: (3616, 10)
Numeric features: ['duration', 'days_left']
Categorical features: ['airline', 'source_city', 'departure_time', 'stops', 'arrival_time', 'destination_city', 'flight_class', 'days_left_group']


In [ ]:

# Preprocessing: scale numerical features and one-hot encode categories
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("poly", PolynomialFeatures(
                    degree=8,
                    include_bias=False
                )),
                ("scaler", StandardScaler())
            ]),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# Train on the training split
baseline_model.fit(X_train, y_train)

# Evaluate on the validation split
val_pred = baseline_model.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, val_pred))
mae = mean_absolute_error(y_val, val_pred)
r2 = r2_score(y_val, val_pred)

print(f"Baseline RMSE: {rmse:.3f}")
print(f"Baseline MAE:  {mae:.3f}")
print(f"Baseline R²:   {r2:.4f}")


Baseline RMSE: 176.260
Baseline MAE:  86.435
Baseline R²:   0.4477


In [ ]:

# Copy the baseline feature tables so we don't modify them
X_flight = X.copy()
X_test_flight = X_test.copy()

# Add flight code
X_flight["flight"] = train.loc[X.index, "flight"]
X_test_flight["flight"] = test["flight"].values

# Add source-destination route
X_flight["route"] = (
    X_flight["source_city"].astype(str)
    + "_to_"
    + X_flight["destination_city"].astype(str)
)

X_test_flight["route"] = (
    X_test_flight["source_city"].astype(str)
    + "_to_"
    + X_test_flight["destination_city"].astype(str)
)

print("Training features:", X_flight.shape)
print("Test features:", X_test_flight.shape)

print("\nUnique flight codes:", X_flight["flight"].nunique())
print("Unique routes:", X_flight["route"].nunique())

print("\nMost common routes:")
print(X_flight["route"].value_counts().head(10))


Training features: (18076, 12)
Test features: (4520, 12)

Unique flight codes: 1123
Unique routes: 18

Most common routes:
route
San Francisco_to_Los Angeles    2065
Los Angeles_to_San Francisco    1995
San Francisco_to_San Diego      1305
San Diego_to_San Francisco      1257
Sacramento_to_Los Angeles        990
Los Angeles_to_Sacramento        977
Sacramento_to_San Jose           872
Los Angeles_to_San Diego         847
San Diego_to_Sacramento          847
Sacramento_to_San Diego          838
Name: count, dtype: int64


USING FLIGHT+ROUTE

In [ ]:

# Re-identify feature types after adding flight and route
numeric_features_flight = X_flight.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features_flight = X_flight.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

# Preprocessing
preprocessor_flight = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("poly", PolynomialFeatures(
                    degree=8,
                    include_bias=False
                )),
                ("scaler", StandardScaler())
            ]),
            numeric_features_flight
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=5
            ),
            categorical_features_flight
        )
    ]
)

# Pipeline
flight_model = Pipeline([
    ("preprocessor", preprocessor_flight),
    ("regressor", LinearRegression())
])

# Use the exact same rows in the train-validation split
X_train_flight = X_flight.loc[X_train.index]
X_val_flight = X_flight.loc[X_val.index]

# Train and evaluate
flight_model.fit(X_train_flight, y_train)

flight_pred = flight_model.predict(X_val_flight)

flight_rmse = np.sqrt(mean_squared_error(y_val, flight_pred))
flight_mae = mean_absolute_error(y_val, flight_pred)
flight_r2 = r2_score(y_val, flight_pred)

print(f"Flight + route RMSE: {flight_rmse:.3f}")
print(f"Flight + route MAE:  {flight_mae:.3f}")
print(f"Flight + route R²:   {flight_r2:.4f}")


Flight + route RMSE: 162.300
Flight + route MAE:  73.114
Flight + route R²:   0.5318


USING ONLY FLIGHT

In [ ]:

# Start with the original feature set and add only flight
X_flight_only = X.copy()
X_flight_only["flight"] = train.loc[X.index, "flight"]

# Identify feature types
num_flight_only = X_flight_only.select_dtypes(
    include=["number"]
).columns.tolist()

cat_flight_only = X_flight_only.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

# Build the preprocessing pipeline
preprocessor_flight_only = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("poly", PolynomialFeatures(
                degree=8,
                include_bias=False
            )),
            ("scaler", StandardScaler())
        ]),
        num_flight_only
    ),
    (
        "cat",
        OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=5
        ),
        cat_flight_only
    )
])

flight_only_model = Pipeline([
    ("preprocessor", preprocessor_flight_only),
    ("regressor", LinearRegression())
])

# Preserve the exact validation rows
X_train_flight_only = X_flight_only.loc[X_train.index]
X_val_flight_only = X_flight_only.loc[X_val.index]

flight_only_model.fit(X_train_flight_only, y_train)

pred_flight_only = flight_only_model.predict(X_val_flight_only)

print(f"Flight-only RMSE: {np.sqrt(mean_squared_error(y_val, pred_flight_only)):.3f}")
print(f"Flight-only MAE:  {mean_absolute_error(y_val, pred_flight_only):.3f}")
print(f"Flight-only R²:   {r2_score(y_val, pred_flight_only):.4f}")


Flight-only RMSE: 163.592
Flight-only MAE:  75.441
Flight-only R²:   0.5243


ROUTE ALONE

In [ ]:

# Start with the original features and add only route
X_route_only = X.copy()

X_route_only["route"] = (
    X_route_only["source_city"].astype(str)
    + "_to_"
    + X_route_only["destination_city"].astype(str)
)

# Identify feature types
num_route = X_route_only.select_dtypes(
    include=["number"]
).columns.tolist()

cat_route = X_route_only.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

# Preprocessing
preprocessor_route_only = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("poly", PolynomialFeatures(
                degree=8,
                include_bias=False
            )),
            ("scaler", StandardScaler())
        ]),
        num_route
    ),
    (
        "cat",
        OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=5
        ),
        cat_route
    )
])

route_only_model = Pipeline([
    ("preprocessor", preprocessor_route_only),
    ("regressor", LinearRegression())
])

X_train_route = X_route_only.loc[X_train.index]
X_val_route = X_route_only.loc[X_val.index]

route_only_model.fit(X_train_route, y_train)

pred_route_only = route_only_model.predict(X_val_route)

print(f"Route-only RMSE: {np.sqrt(mean_squared_error(y_val, pred_route_only)):.3f}")
print(f"Route-only MAE:  {mean_absolute_error(y_val, pred_route_only):.3f}")
print(f"Route-only R²:   {r2_score(y_val, pred_route_only):.4f}")


Route-only RMSE: 172.892
Route-only MAE:  80.597
Route-only R²:   0.4686


RANDOM FOREST APPROACH

In [ ]:

from sklearn.ensemble import RandomForestRegressor

# Use the best feature set found so far
X_rf = X_flight.copy()

# Identify feature types
numeric_rf = X_rf.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_rf = X_rf.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

# Trees do not need polynomial expansion or standard scaling
preprocessor_rf = ColumnTransformer(
    transformers=[
        (
            "num",
            "passthrough",
            numeric_rf
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=5
            ),
            categorical_rf
        )
    ]
)

rf_model = Pipeline([
    ("preprocessor", preprocessor_rf),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
        n_jobs=-1
    ))
])

# Same train-validation rows as our previous experiments
X_train_rf = X_rf.loc[X_train.index]
X_val_rf = X_rf.loc[X_val.index]

rf_model.fit(X_train_rf, y_train)

rf_pred = rf_model.predict(X_val_rf)

rf_rmse = np.sqrt(mean_squared_error(y_val, rf_pred))
rf_mae = mean_absolute_error(y_val, rf_pred)
rf_r2 = r2_score(y_val, rf_pred)

print(f"Random Forest RMSE: {rf_rmse:.3f}")
print(f"Random Forest MAE:  {rf_mae:.3f}")
print(f"Random Forest R²:   {rf_r2:.4f}")



Random Forest RMSE: 145.500
Random Forest MAE:  45.695
Random Forest R²:   0.6237


WE ARE KEEPING 1,3,5 ON LEAF

In [ ]:

from sklearn.ensemble import RandomForestRegressor

leaf_results = []

for leaf_size in [1, 3, 5]:
    model = Pipeline([
        ("preprocessor", preprocessor_rf),
        ("regressor", RandomForestRegressor(
            n_estimators=200,
            min_samples_leaf=leaf_size,
            max_features=0.8,
            random_state=42,
            n_jobs=-1
        ))
    ])

    model.fit(X_train_rf, y_train)
    predictions = model.predict(X_val_rf)

    rmse = np.sqrt(mean_squared_error(y_val, predictions))
    mae = mean_absolute_error(y_val, predictions)
    r2 = r2_score(y_val, predictions)

    leaf_results.append({
        "min_samples_leaf": leaf_size,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

    print(
        f"Leaf={leaf_size} | "
        f"RMSE={rmse:.3f} | "
        f"MAE={mae:.3f} | "
        f"R²={r2:.4f}"
    )

leaf_results_df = pd.DataFrame(leaf_results)
print("\nResults sorted by RMSE:")
print(leaf_results_df.sort_values("RMSE").to_string(index=False))


Leaf=1 | RMSE=146.730 | MAE=45.167 | R²=0.6173
Leaf=3 | RMSE=145.500 | MAE=45.695 | R²=0.6237
Leaf=5 | RMSE=145.225 | MAE=47.045 | R²=0.6251

Results sorted by RMSE:
 min_samples_leaf       RMSE       MAE       R2
                5 145.225439 47.044537 0.625098
                3 145.499804 45.695372 0.623680
                1 146.730227 45.167201 0.617288


TUNING max_feature

In [ ]:
feature_results = []

for feature_fraction in [0.6, 0.8, 1.0]:
    model = Pipeline([
        ("preprocessor", preprocessor_rf),
        ("regressor", RandomForestRegressor(
            n_estimators=200,
            min_samples_leaf=5,
            max_features=feature_fraction,
            random_state=42,
            n_jobs=-1
        ))
    ])

    model.fit(X_train_rf, y_train)
    predictions = model.predict(X_val_rf)

    rmse = np.sqrt(mean_squared_error(y_val, predictions))
    mae = mean_absolute_error(y_val, predictions)
    r2 = r2_score(y_val, predictions)

    feature_results.append({
        "max_features": feature_fraction,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

    print(
        f"max_features={feature_fraction} | "
        f"RMSE={rmse:.3f} | "
        f"MAE={mae:.3f} | "
        f"R²={r2:.4f}"
    )

print("\nResults sorted by RMSE:")
print(
    pd.DataFrame(feature_results)
    .sort_values("RMSE")
    .to_string(index=False)
)

max_features=0.6 | RMSE=144.937 | MAE=47.385 | R²=0.6266
max_features=0.8 | RMSE=145.225 | MAE=47.045 | R²=0.6251
max_features=1.0 | RMSE=145.541 | MAE=46.948 | R²=0.6235

Results sorted by RMSE:
 max_features       RMSE       MAE       R2
          0.6 144.936919 47.385274 0.626586
          0.8 145.225439 47.044537 0.625098
          1.0 145.541331 46.947612 0.623465


In [ ]:
tree_results = []

for n_trees in [100, 200, 400]:
    model = Pipeline([
        ("preprocessor", preprocessor_rf),
        ("regressor", RandomForestRegressor(
            n_estimators=n_trees,
            min_samples_leaf=5,
            max_features=0.6,
            random_state=42,
            n_jobs=-1
        ))
    ])

    model.fit(X_train_rf, y_train)
    predictions = model.predict(X_val_rf)

    rmse = np.sqrt(mean_squared_error(y_val, predictions))
    mae = mean_absolute_error(y_val, predictions)
    r2 = r2_score(y_val, predictions)

    tree_results.append({
        "n_estimators": n_trees,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

    print(
        f"Trees={n_trees} | "
        f"RMSE={rmse:.3f} | "
        f"MAE={mae:.3f} | "
        f"R²={r2:.4f}"
    )

print("\nResults sorted by RMSE:")
print(
    pd.DataFrame(tree_results)
    .sort_values("RMSE")
    .to_string(index=False)
)

Trees=100 | RMSE=145.380 | MAE=47.452 | R²=0.6243
Trees=200 | RMSE=144.937 | MAE=47.385 | R²=0.6266
Trees=400 | RMSE=144.754 | MAE=47.240 | R²=0.6275

Results sorted by RMSE:
 n_estimators       RMSE       MAE       R2
          400 144.754113 47.240299 0.627527
          200 144.936919 47.385274 0.626586
          100 145.379734 47.451891 0.624301


GRADIENT BOOSTING (will not be used becoz bad compared to random forest)

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb_model = Pipeline([
    ("preprocessor", preprocessor_rf),
    ("regressor", GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        min_samples_leaf=5,
        loss="squared_error",
        random_state=42
    ))
])

gb_model.fit(X_train_rf, y_train)

gb_pred = gb_model.predict(X_val_rf)

gb_rmse = np.sqrt(mean_squared_error(y_val, gb_pred))
gb_mae = mean_absolute_error(y_val, gb_pred)
gb_r2 = r2_score(y_val, gb_pred)

print(f"Gradient Boosting RMSE: {gb_rmse:.3f}")
print(f"Gradient Boosting MAE:  {gb_mae:.3f}")
print(f"Gradient Boosting R²:   {gb_r2:.4f}")

Gradient Boosting RMSE: 157.129
Gradient Boosting MAE:  65.720
Gradient Boosting R²:   0.5611


In [ ]:
gb_results = []

for depth, n_trees in [(3, 400), (4, 400), (5, 400)]:
    model = Pipeline([
        ("preprocessor", preprocessor_rf),
        ("regressor", GradientBoostingRegressor(
            n_estimators=n_trees,
            learning_rate=0.05,
            max_depth=depth,
            min_samples_leaf=5,
            loss="squared_error",
            random_state=42
        ))
    ])

    model.fit(X_train_rf, y_train)
    predictions = model.predict(X_val_rf)

    rmse = np.sqrt(mean_squared_error(y_val, predictions))
    mae = mean_absolute_error(y_val, predictions)
    r2 = r2_score(y_val, predictions)

    gb_results.append({
        "max_depth": depth,
        "n_estimators": n_trees,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

    print(
        f"Depth={depth}, Trees={n_trees} | "
        f"RMSE={rmse:.3f} | MAE={mae:.3f} | R²={r2:.4f}"
    )

print("\nSorted by RMSE:")
print(
    pd.DataFrame(gb_results)
    .sort_values("RMSE")
    .to_string(index=False)
)

Depth=3, Trees=400 | RMSE=154.673 | MAE=62.529 | R²=0.5747
Depth=4, Trees=400 | RMSE=148.686 | MAE=57.566 | R²=0.6070
Depth=5, Trees=400 | RMSE=149.281 | MAE=54.965 | R²=0.6039

Sorted by RMSE:
 max_depth  n_estimators       RMSE       MAE       R2
         4           400 148.685677 57.566319 0.607020
         5           400 149.280969 54.964892 0.603867
         3           400 154.673442 62.529306 0.574731


BACK TO RANDOM FOREST  

In [ ]:
# Train the best Random Forest configuration
best_rf = Pipeline([
    ("preprocessor", preprocessor_rf),
    ("regressor", RandomForestRegressor(
        n_estimators=400,
        min_samples_leaf=5,
        max_features=0.6,
        random_state=42,
        n_jobs=-1
    ))
])

best_rf.fit(X_train_rf, y_train)

# Extract transformed feature names and importances
feature_names = (
    best_rf.named_steps["preprocessor"].get_feature_names_out()
)

importances = best_rf.named_steps["regressor"].feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

print("Top 20 features:")
print(importance_df.head(20).to_string(index=False))

Top 20 features:
                          Feature  Importance
                    num__duration    0.229174
        cat__flight_class_Economy    0.125741
       cat__flight_class_Business    0.113488
                   num__days_left    0.111916
                  cat__stops_zero    0.059985
              cat__flight_UA 1055    0.036711
              cat__airline_United    0.035463
              cat__flight_UA 2013    0.021529
cat__route_Sacramento_to_San Jose    0.020794
            cat__airline_Frontier    0.020118
   cat__departure_time_Late_Night    0.016555
                   cat__stops_one    0.016046
     cat__airline_Alaska_Airlines    0.012352
cat__route_San Jose_to_Sacramento    0.012223
   cat__airline_American_Airlines    0.009128
         cat__days_left_group_1-7    0.008626
   cat__destination_city_San Jose    0.008348
   cat__flight_infrequent_sklearn    0.007138
       cat__days_left_group_31-45    0.006092
       cat__source_city_San Diego    0.006054


In [ ]:
importance_df["Original_Feature"] = (
    importance_df["Feature"]
    .str.replace(r"^num__", "", regex=True)
    .str.replace(r"^cat__", "", regex=True)
    .str.split("_")
    .str[0]
)

# Better grouping for categorical columns:
def get_original_feature(name):
    if name.startswith("num__"):
        return name.replace("num__", "")
    if name.startswith("cat__"):
        name = name.replace("cat__", "")
        for col in categorical_features_flight:
            if name == col or name.startswith(col + "_"):
                return col
    return name

importance_df["Original_Feature"] = importance_df["Feature"].apply(
    get_original_feature
)

grouped_importance = (
    importance_df.groupby("Original_Feature")["Importance"]
    .sum()
    .sort_values(ascending=False)
)

print("Importance by original feature:")
print((grouped_importance * 100).round(2).to_string())

Importance by original feature:
Original_Feature
flight_class        23.92
duration            22.92
days_left           11.19
flight               9.45
airline              8.44
stops                7.67
route                6.13
departure_time       2.74
days_left_group      2.17
destination_city     1.98
source_city          1.82
arrival_time         1.58


In [ ]:
#running current random forest with and without days_left_group derived from days_left
from sklearn.base import clone

X_train_no_group = X_train_rf.drop(columns=["days_left_group"])
X_val_no_group = X_val_rf.drop(columns=["days_left_group"])

rf_with_group = Pipeline([
    ("preprocessor", preprocessor_rf),
    ("regressor", RandomForestRegressor(
        n_estimators=400,
        min_samples_leaf=5,
        max_features=0.6,
        random_state=42,
        n_jobs=-1
    ))
])

preprocessor_no_group = clone(preprocessor_rf)
preprocessor_no_group.transformers = [
    (name, transformer,
     [col for col in columns if col != "days_left_group"])
    for name, transformer, columns in preprocessor_rf.transformers
]

rf_without_group = Pipeline([
    ("preprocessor", preprocessor_no_group),
    ("regressor", RandomForestRegressor(
        n_estimators=400,
        min_samples_leaf=5,
        max_features=0.6,
        random_state=42,
        n_jobs=-1
    ))
])

rf_with_group.fit(X_train_rf, y_train)
pred_with = rf_with_group.predict(X_val_rf)

rf_without_group.fit(X_train_no_group, y_train)
pred_without = rf_without_group.predict(X_val_no_group)

print("WITH days_left_group")
print("RMSE:", round(np.sqrt(mean_squared_error(y_val, pred_with)), 3))
print("MAE: ", round(mean_absolute_error(y_val, pred_with), 3))
print("R²:  ", round(r2_score(y_val, pred_with), 4))

print("\nWITHOUT days_left_group")
print("RMSE:", round(np.sqrt(mean_squared_error(y_val, pred_without)), 3))
print("MAE: ", round(mean_absolute_error(y_val, pred_without), 3))
print("R²:  ", round(r2_score(y_val, pred_without), 4))

WITH days_left_group
RMSE: 144.754
MAE:  47.24
R²:   0.6275

WITHOUT days_left_group
RMSE: 144.739
MAE:  47.017
R²:   0.6276


In [ ]:
#running current random forest with and without flight feature

# Model 1: WITH flight
X_with_flight = X_flight.drop(columns=["days_left_group"])
X_train_with = X_with_flight.loc[X_train.index]
X_val_with = X_with_flight.loc[X_val.index]

# Model 2: WITHOUT flight
X_without_flight = X_with_flight.drop(columns=["flight"])
X_train_without = X_without_flight.loc[X_train.index]
X_val_without = X_without_flight.loc[X_val.index]


def evaluate_rf(X_tr, X_v, y_tr, y_v):
    numeric_cols = X_tr.select_dtypes(include=["number"]).columns.tolist()
    categorical_cols = X_tr.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    preprocessor = ColumnTransformer([
        ("num", "passthrough", numeric_cols),
        ("cat", OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=5
        ), categorical_cols)
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(
            n_estimators=400,
            min_samples_leaf=5,
            max_features=0.6,
            random_state=42,
            n_jobs=-1
        ))
    ])

    model.fit(X_tr, y_tr)
    predictions = model.predict(X_v)

    return {
        "RMSE": np.sqrt(mean_squared_error(y_v, predictions)),
        "MAE": mean_absolute_error(y_v, predictions),
        "R2": r2_score(y_v, predictions)
    }


results_with = evaluate_rf(
    X_train_with, X_val_with, y_train, y_val
)

results_without = evaluate_rf(
    X_train_without, X_val_without, y_train, y_val
)

comparison = pd.DataFrame(
    [results_with, results_without],
    index=["With flight", "Without flight"]
)

print(comparison.round(4))

                    RMSE      MAE      R2
With flight     144.7392  47.0169  0.6276
Without flight  145.4611  47.8781  0.6239


In [ ]:
#running current random forest with and without flight feature

# Model 1: WITH route
X_with_route = X_flight.drop(columns=["days_left_group"])
X_train_with_route = X_with_route.loc[X_train.index]
X_val_with_route = X_with_route.loc[X_val.index]

# Model 2: WITHOUT route
X_without_route = X_with_route.drop(columns=["route"])
X_train_without_route = X_without_route.loc[X_train.index]
X_val_without_route = X_without_route.loc[X_val.index]


results_with_route = evaluate_rf(
    X_train_with_route, X_val_with_route, y_train, y_val
)

results_without_route = evaluate_rf(
    X_train_without_route, X_val_without_route, y_train, y_val
)

route_comparison = pd.DataFrame(
    [results_with_route, results_without_route],
    index=["With route", "Without route"]
)

print(route_comparison.round(4))

                   RMSE      MAE      R2
With route     144.7392  47.0169  0.6276
Without route  145.3049  48.2469  0.6247


**TRAINING MODEL ON RANDOM FOREST WITH FULL TRAIN DATA**

In [ ]:
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

# 1. Prepare final training and test features
X_final = X_flight.drop(columns=["days_left_group"])
X_test_final = X_test_flight.drop(columns=["days_left_group"])

y_final = train["price"]

# Ensure training and test columns match
X_test_final = X_test_final[X_final.columns]

numeric_cols = X_final.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_final.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Training features:", X_final.shape)
print("Test features:", X_test_final.shape)

# 2. Build the final model
final_preprocessor = ColumnTransformer([
    ("num", "passthrough", numeric_cols),
    ("cat", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=5
    ), categorical_cols)
])

final_model = Pipeline([
    ("preprocessor", final_preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=400,
        min_samples_leaf=5,
        max_features=0.6,
        random_state=42,
        n_jobs=-1
    ))
])

# 3. Train on ALL available training rows
final_model.fit(X_final, y_final)

print("Final model training complete!")

# 4. Predict prices for the test dataset
test_predictions = final_model.predict(X_test_final)

print("Number of predictions:", len(test_predictions))
print("Sample predictions:", test_predictions[:10])

# 5. Populate the Kaggle submission template
final_submission = submission.copy()
final_submission["price"] = test_predictions

# 6. Save the submission and trained model
submission_path = f"{DATA_DIR}/final_submission.csv"
model_path = f"{DATA_DIR}/final_random_forest.joblib"

final_submission.to_csv(submission_path, index=False)
joblib.dump(final_model, model_path)

print("\nSubmission shape:", final_submission.shape)
print("Submission columns:", final_submission.columns.tolist())
print("Missing predictions:", final_submission["price"].isna().sum())
print("Submission saved to:", submission_path)
print("Model saved to:", model_path)

display(final_submission.head())

Training features: (18076, 11)
Test features: (4520, 11)
Final model training complete!
Number of predictions: 4520
Sample predictions: [ 68.94630437 163.33593609 433.16232763 140.36888394 507.5171979
 216.25434844 107.59947586 153.16966962 142.07448995 194.10789575]

Submission shape: (4520, 2)
Submission columns: ['id', 'price']
Missing predictions: 0
Submission saved to: /content/drive/MyDrive/ML_project_dataset/final_submission.csv
Model saved to: /content/drive/MyDrive/ML_project_dataset/final_random_forest.joblib


,id,price
0,18395,68.946304
1,11771,163.335936
2,22326,433.162328
3,18332,140.368884
4,11021,507.517198


In [ ]:
# Verify submission structure and values
assert len(final_submission) == len(test), "Wrong number of rows!"
assert final_submission.columns.tolist() == submission.columns.tolist(), "Column mismatch!"
assert final_submission["id"].equals(submission["id"]), "ID order mismatch!"
assert final_submission["price"].notna().all(), "Missing predictions!"
assert np.isfinite(final_submission["price"]).all(), "Invalid numeric predictions!"

print("All submission checks passed!")
print("\nPrediction summary:")
print(final_submission["price"].describe())

print("\nFirst 5 rows:")
display(final_submission.head())

All submission checks passed!

Prediction summary:
count    4520.000000
mean      281.223452
std       185.141797
min        38.538574
25%       160.823188
50%       242.252284
75%       349.201137
max      2034.105629
Name: price, dtype: float64

First 5 rows:


,id,price
0,18395,68.946304
1,11771,163.335936
2,22326,433.162328
3,18332,140.368884
4,11021,507.517198


In [ ]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/ML_project_dataset/final_submission.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
import pandas as pd
from IPython.display import display

lr_optimization_results = [
    {"Experiment": "Polynomial degree 8 baseline", "Evaluation": "80/20 holdout", "MAE": 86.435, "RMSE": 176.260, "R2": 0.4477},
    {"Experiment": "Add flight + route", "Evaluation": "80/20 holdout", "MAE": 73.114, "RMSE": 162.300, "R2": 0.5318},
    {"Experiment": "Add flight only", "Evaluation": "80/20 holdout", "MAE": 75.441, "RMSE": 163.592, "R2": 0.5243},
    {"Experiment": "Add route only", "Evaluation": "80/20 holdout", "MAE": 80.597, "RMSE": 172.892, "R2": 0.4686},
    {"Experiment": "Random Forest initial configuration", "Evaluation": "80/20 holdout", "MAE": 45.695, "RMSE": 145.500, "R2": 0.6237},
    {"Experiment": "Random Forest: 200 trees, leaf=5, max_features=0.6", "Evaluation": "80/20 holdout", "MAE": 47.385, "RMSE": 144.937, "R2": 0.6266},
    {"Experiment": "Random Forest: 400 trees, leaf=5, max_features=0.6", "Evaluation": "80/20 holdout", "MAE": 47.240, "RMSE": 144.754, "R2": 0.6275},
    {"Experiment": "Gradient Boosting: 200 trees, depth=3", "Evaluation": "80/20 holdout", "MAE": 65.720, "RMSE": 157.129, "R2": 0.5611},
    {"Experiment": "Gradient Boosting: 400 trees, depth=3", "Evaluation": "80/20 holdout", "MAE": 62.529, "RMSE": 154.673, "R2": 0.5747},
    {"Experiment": "Gradient Boosting: 400 trees, depth=4", "Evaluation": "80/20 holdout", "MAE": 57.566, "RMSE": 148.686, "R2": 0.6070},
    {"Experiment": "Gradient Boosting: 400 trees, depth=5", "Evaluation": "80/20 holdout", "MAE": 54.965, "RMSE": 149.281, "R2": 0.6039},
]
lr_optimization_results_df = pd.DataFrame(lr_optimization_results).sort_values("RMSE")
display(lr_optimization_results_df.round(4))

,Experiment,Evaluation,MAE,RMSE,R2
6,"Random Forest: 400 trees, leaf=5, max_features...",80/20 holdout,47.240,144.754,0.6275
5,"Random Forest: 200 trees, leaf=5, max_features...",80/20 holdout,47.385,144.937,0.6266
4,Random Forest initial configuration,80/20 holdout,45.695,145.500,0.6237
9,"Gradient Boosting: 400 trees, depth=4",80/20 holdout,57.566,148.686,0.6070
10,"Gradient Boosting: 400 trees, depth=5",80/20 holdout,54.965,149.281,0.6039
8,"Gradient Boosting: 400 trees, depth=3",80/20 holdout,62.529,154.673,0.5747
7,"Gradient Boosting: 200 trees, depth=3",80/20 holdout,65.720,157.129,0.5611
1,Add flight + route,80/20 holdout,73.114,162.300,0.5318
2,Add flight only,80/20 holdout,75.441,163.592,0.5243
3,Add route only,80/20 holdout,80.597,172.892,0.4686
